# Lakelink Fiber PRI: v4 candidate writers (@ml-engineer)

**Task:** P37-ML-V4-WRITERS-CHECKIN. This is the versioned writer for the two ML shadow objects that P34-ML-V4-RESCORE-DRYRUN built in an ad-hoc session on 2026-10-25 around 21:00 UTC:

| Target | Grain | Built from |
| --- | --- | --- |
| `span_risk_predictions_v4_candidate` | 1 row per span (10,092) | UC `span_risk_classifier@champion` (v7). 7 hazard features come from in-radius pairs in `hazard_span_intersection_v4_candidate`. The other 28 come from `span_risk_inference_log`. Includes a v3@22 rule control and the production class. |
| `resilience_score_snapshot_v4_candidate` | 1 row per service (8,000) | V2 formula (HL20, SLA boost, capped at 100) with active hazards from the v4 HSI at the v3 score_time. Diversity and telemetry are carried over from FROZEN v3 snapshot v22. |

**Freeze rules (DEC-PRI-V4-SHADOW-BUILD):** writes go only to `*_v4_candidate` targets. FROZEN v3 objects are only read, through `VERSION AS OF`. `WRITE_ENABLED` defaults to `false`, so a normal run builds the frames and checks parity against the live candidate tables without writing anything. `INSERT OVERWRITE` keeps the existing table and column comments.

**Parity proof (S38, 2026-10-26, read-only):** predictions class 10,092/10,092 (v4) and 10,092/10,092 (v3rule control). Hazard counts 10,092/10,092. Snapshot active_hazards 8,000/8,000, hazard_exposure 8,000/8,000, overall_score 7,999/8,000 (one float tie off by 0.01), risk_band 8,000/8,000. KPIs: avg 56.77, critical 1,087, revenue at risk $54.44M.

In [0]:
import mlflow
import pandas as pd
from pyspark.sql import functions as F

dbutils.widgets.dropdown("WRITE_ENABLED", "false", ["false", "true"], "Write to *_v4_candidate")
dbutils.widgets.text("LIFT_TASK", "", "Approved task code (optional)")
WRITE_ENABLED = dbutils.widgets.get("WRITE_ENABLED") == "true"
LIFT_TASK = dbutils.widgets.get("LIFT_TASK") or None

SCHEMA = "cdm_tmforum.provable_resilience_intelligence"
MODEL_NAME = f"{SCHEMA}.span_risk_classifier"
MODEL_ALIAS = "champion"

# Inputs (read only)
HSI_V4 = f"{SCHEMA}.hazard_span_intersection_v4_candidate"
HSI_V3_PIN = 22
SNAPSHOT_V3_PIN = 22
INFERENCE_LOG = f"{SCHEMA}.span_risk_inference_log"
SERVICE_INVENTORY = f"{SCHEMA}.service_inventory"

# Outputs (shadow only)
OUT_PRED = f"{SCHEMA}.span_risk_predictions_v4_candidate"
OUT_SNAP = f"{SCHEMA}.resilience_score_snapshot_v4_candidate"

# Held at the v3 snapshot v22 score_time so the deltas are like-for-like
SCORE_TIME = "2026-10-24 13:08:47"
FEATURE_SOURCE = "hazard_span_intersection_v4_candidate@v1 (in-radius pairs by type) + span_risk_inference_log (28 non-hazard features)"

FEATURES = [
    "fiber_count", "wavelength_count", "length_km", "span_age_years",
    "total_alarms", "distinct_alarm_types", "critical_alarm_count", "major_alarm_count",
    "severe_alarm_count", "osnr_degrade_count", "ber_high_count", "fiber_cut_count", "los_count",
    "avg_alarm_duration_hrs", "unack_alarm_count", "days_since_last_critical",
    "srlg_count", "srlg_type_count", "srlg_critical_count", "srlg_severe_count",
    "service_count", "spans_in_conduit",
    "wildfire_count", "flood_count", "dig_ticket_count", "weather_count", "earthquake_count",
    "hazard_total_count", "hazard_critical_count",
    "avg_optical_power_dbm", "avg_osnr_db", "avg_ber", "avg_pmd_ps", "stddev_osnr", "stddev_ber",
]
HAZARD_FEATURES = ["wildfire_count", "flood_count", "dig_ticket_count", "weather_count",
                   "earthquake_count", "hazard_total_count", "hazard_critical_count"]
assert len(FEATURES) == 35
print(f"WRITE_ENABLED={WRITE_ENABLED} | outputs: {OUT_PRED}, {OUT_SNAP}")

In [0]:
def hazard_features(source_sql: str):
    """In-radius hazard pairs per span, by hazard type. Critical = hazard_severity critical.

    This is the documented rule from P34. The original v3 hazard feature code was lost
    with the deleted experiment, so the same rule is applied to the v4 HSI (treatment)
    and to HSI@22 (control).
    """
    return spark.sql(f"""
        SELECT span_id,
               count(*)                                      AS hazard_total_count,
               sum(int(hazard_type = 'wildfire'))            AS wildfire_count,
               sum(int(hazard_type = 'flood'))               AS flood_count,
               sum(int(hazard_type = 'dig_ticket'))          AS dig_ticket_count,
               sum(int(hazard_type = 'severe_weather'))      AS weather_count,
               sum(int(hazard_type = 'earthquake'))          AS earthquake_count,
               sum(int(lower(hazard_severity) = 'critical')) AS hazard_critical_count
        FROM {source_sql}
        WHERE is_within_radius
        GROUP BY span_id""")

hz_v4 = hazard_features(HSI_V4)
hz_v3 = hazard_features(f"{SCHEMA}.hazard_span_intersection VERSION AS OF {HSI_V3_PIN}")

log = spark.table(INFERENCE_LOG)
base_cols = [c for c in FEATURES if c not in HAZARD_FEATURES]

def feature_frame(hz):
    return (log.select("span_id", *base_cols, F.col("predicted_risk_class").alias("v3_prod_risk_class"))
               .join(hz, "span_id", "left")
               .fillna(0, subset=HAZARD_FEATURES)
               .toPandas())

pdf_v4 = feature_frame(hz_v4)
pdf_v3 = feature_frame(hz_v3)
print(len(pdf_v4), "spans |", int(pdf_v4.hazard_total_count.sum()), "v4 pairs |", int(pdf_v3.hazard_total_count.sum()), "v3@22 pairs")

In [0]:
mlflow.set_registry_uri("databricks-uc")
mv = mlflow.MlflowClient().get_model_version_by_alias(MODEL_NAME, MODEL_ALIAS)
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@{MODEL_ALIAS}")  # needs xgboost==3.4.1

def predict(pdf):
    out = model.predict(pdf[FEATURES].astype("float64"))
    return out if isinstance(out, pd.DataFrame) else pd.DataFrame(out)

pred_v4 = predict(pdf_v4)
pred_v3 = predict(pdf_v3)

out = pd.DataFrame({
    "span_id": pdf_v4.span_id.values,
    "predicted_risk_class": pred_v4["predicted_risk_class"].astype("int64").values,
    "predicted_risk_label": pred_v4["predicted_risk_label"].values,
    "prob_low": pred_v4["prob_low"].values,
    "prob_moderate": pred_v4["prob_moderate"].values,
    "prob_high": pred_v4["prob_high"].values,
    "prob_critical": pred_v4["prob_critical"].values,
    "hazard_total_count_v4": pdf_v4.hazard_total_count.astype("int64").values,
    "v3_prod_risk_class": pdf_v4.v3_prod_risk_class.astype("int64").values,
})
ctrl = pd.DataFrame({
    "span_id": pdf_v3.span_id.values,
    "v3rule_risk_class": pred_v3["predicted_risk_class"].astype("int64").values,
    "hazard_total_count_v3rule": pdf_v3.hazard_total_count.astype("int64").values,
})
out = out.merge(ctrl, on="span_id", how="left")

pred_sdf = (spark.createDataFrame(out)
    .withColumn("model_version", F.lit(f"uc_v{mv.version}_champion"))
    .withColumn("model_run_id", F.lit(mv.run_id))
    .withColumn("origin", F.lit("INFERENCE"))
    .withColumn("feature_source", F.lit(FEATURE_SOURCE))
    .withColumn("scored_at", F.current_timestamp())
    .select("span_id", "predicted_risk_class", "predicted_risk_label", "prob_low", "prob_moderate",
            "prob_high", "prob_critical", "v3rule_risk_class", "v3_prod_risk_class",
            "hazard_total_count_v4", "hazard_total_count_v3rule", "model_version", "model_run_id",
            "origin", "feature_source", "scored_at"))
pred_sdf.createOrReplaceTempView("v4_pred_stage")
print(f"champion v{mv.version} | critical spans v4={int((out.predicted_risk_class == 3).sum())} v3rule={int((out.v3rule_risk_class == 3).sum())}")

In [0]:
# Banker's rounding (bround) and an unrounded exposure inside overall_score reproduce the
# P34 pandas output (see the parity cell). Active hazard = in radius, started at or before
# SCORE_TIME, and not yet ended.
snap_sdf = spark.sql(f"""
WITH hz AS (
  SELECT DISTINCT trim(sid) AS service_id, hazard_id
  FROM {HSI_V4}
  LATERAL VIEW explode(split(affected_service_ids, ',')) e AS sid
  WHERE is_within_radius
    AND hazard_start <= TIMESTAMP'{SCORE_TIME}'
    AND (hazard_end IS NULL OR hazard_end > TIMESTAMP'{SCORE_TIME}')
),
ah AS (SELECT service_id, count(DISTINCT hazard_id) AS active_hazards FROM hz GROUP BY service_id),
v3 AS (SELECT * FROM {SCHEMA}.resilience_score_snapshot VERSION AS OF {SNAPSHOT_V3_PIN}),
b AS (
  SELECT v3.service_id, v3.score_time, v3.path_diversity_score, v3.telemetry_health_score,
         v3.srlg_count, v3.active_alarms, si.sla_tier, si.mrc_usd,
         coalesce(ah.active_hazards, 0) AS active_hazards,
         v3.active_hazards AS v3_active_hazards, v3.hazard_exposure_score AS v3_hazard_exposure_score,
         v3.overall_score AS v3_overall_score, v3.revenue_at_risk_usd AS v3_revenue_at_risk_usd
  FROM v3 JOIN {SERVICE_INVENTORY} si USING (service_id) LEFT JOIN ah USING (service_id)
),
s AS (
  SELECT *,
    bround(100 * exp(-active_hazards * ln(2) / 20), 2) AS hazard_exposure_score,
    bround(least(100, 0.4 * path_diversity_score + 0.3 * (100 * exp(-active_hazards * ln(2) / 20))
                 + 0.3 * telemetry_health_score
                 + CASE sla_tier WHEN 'premium_plus' THEN 5 WHEN 'premium' THEN 2 ELSE 0 END), 2) AS overall_score
  FROM b
)
SELECT service_id, score_time, path_diversity_score, telemetry_health_score, srlg_count, active_alarms,
       sla_tier, mrc_usd, active_hazards, hazard_exposure_score, overall_score,
       bround(mrc_usd * (1 - overall_score / 100), 2) AS revenue_at_risk_usd,
       v3_active_hazards, v3_hazard_exposure_score, v3_overall_score, v3_revenue_at_risk_usd,
       CASE WHEN overall_score < 40 THEN 'critical' WHEN overall_score < 60 THEN 'high'
            WHEN overall_score < 80 THEN 'medium' WHEN overall_score < 90 THEN 'good' ELSE 'excellent' END AS risk_band,
       CASE WHEN v3_overall_score < 40 THEN 'critical' WHEN v3_overall_score < 60 THEN 'high'
            WHEN v3_overall_score < 80 THEN 'medium' WHEN v3_overall_score < 90 THEN 'good' ELSE 'excellent' END AS v3_risk_band,
       'DERIVED' AS origin, current_timestamp() AS created_at
FROM s""")
snap_sdf.createOrReplaceTempView("v4_snap_stage")
display(snap_sdf.agg(F.count("*").alias("services"), F.round(F.avg("overall_score"), 2).alias("avg_score"),
                     F.sum((F.col("overall_score") < 40).cast("int")).alias("critical"),
                     F.round(F.sum("revenue_at_risk_usd") / 1e6, 2).alias("revenue_at_risk_musd")))

In [0]:
live_pred = spark.table(OUT_PRED).alias("l")
jp = pred_sdf.alias("n").join(live_pred, "span_id", "full")
pred_parity = jp.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("n.predicted_risk_class").eqNullSafe(F.col("l.predicted_risk_class")).cast("int")).alias("class_v4"),
    F.sum(F.col("n.v3rule_risk_class").eqNullSafe(F.col("l.v3rule_risk_class")).cast("int")).alias("class_v3rule"),
    F.sum(F.col("n.hazard_total_count_v4").eqNullSafe(F.col("l.hazard_total_count_v4")).cast("int")).alias("hz_v4"),
).first()

live_snap = spark.table(OUT_SNAP).alias("l")
js = snap_sdf.alias("n").join(live_snap, "service_id", "full")
snap_parity = js.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("n.active_hazards").eqNullSafe(F.col("l.active_hazards")).cast("int")).alias("active_hazards"),
    F.sum((F.abs(F.col("n.overall_score") - F.col("l.overall_score")) <= 0.01).cast("int")).alias("overall_within_0_01"),
    F.sum(F.col("n.risk_band").eqNullSafe(F.col("l.risk_band")).cast("int")).alias("risk_band"),
).first()
print("predictions:", pred_parity.asDict())
print("snapshot:   ", snap_parity.asDict())

In [0]:
if not WRITE_ENABLED:
    print("WRITE_ENABLED=false: dry run only, nothing written.")
else:
    from ml_freeze_guard import assert_writable
    for target in (OUT_PRED, OUT_SNAP):
        assert target.endswith("_v4_candidate"), f"Refusing non-shadow target {target}"
        assert_writable(spark, target.split(".")[-1], lift_task=LIFT_TASK)
    # INSERT OVERWRITE keeps the table and column comments (semantic modeling mandate)
    spark.sql(f"INSERT OVERWRITE TABLE {OUT_PRED} SELECT * FROM v4_pred_stage")
    spark.sql(f"INSERT OVERWRITE TABLE {OUT_SNAP} SELECT * FROM v4_snap_stage")
    print(f"Wrote {spark.table(OUT_PRED).count()} rows to {OUT_PRED}, {spark.table(OUT_SNAP).count()} rows to {OUT_SNAP}")